# (Enter Code's Name)

## Imports

## Circuit Generating Code

In [ ]:
def generate_qec_circuit(distance, rounds, noise_prob):
    """
    Inputs:
    Distance is code distance
    rounds is number of rounds of stabilizers
    noise_prob is the probability of bit flip error in physical qubits
    
    Output: returns circuit
    """
    # Put code here:
    
    return circuit

In [ ]:
def generate_code_bit_flips(d, noise):
    # SOLUTION ===
    circuit = stim.Circuit.generated(
    "repetition_code:memory",
    rounds=3*d,
    distance=d,
    before_measure_flip_probability=noise)
    # ===
    return circuit

In [ ]:
decoder = ['string enter']

## Threshold

In [ ]:
qec_code_tasks = [
    sinter.Task(
        circuit = generate_code_bit_flips(d, noise),
        json_metadata={'d': d, 'r': d * 3, 'p': noise},
    )
    for d in [3, 5, 7, 9]
    for noise in [0.008, 0.009, 0.01, 0.011, 0.012]
]

collected_code_stats: List[sinter.TaskStats] = sinter.collect(
    num_workers=4,
    tasks=qce_code_tasks,
    decoders=decoder,
    max_shots=10**5,
    max_errors=5_000
)

In [ ]:
fig, ax = plt.subplots(1, 1)
sinter.plot_error_rate(
    ax=ax,
    stats=collected_stats,
    x_func=lambda stats: stats.json_metadata['p'],
    group_func=lambda stats: stats.json_metadata['d'],
)

ax.loglog()
ax.set_title("Repetition Code Error Rates (Phenomenological Noise)")
ax.set_xlabel("Phyical Error Rate")
ax.set_ylabel("Logical Error Rate per Shot")
ax.grid(which='major')
ax.grid(which='minor')
ax.legend()
fig.set_dpi(120)  # Show it bigger

## Resources Required to reach $10^{-10}$ bit flip error

In [ ]:
bit_flip_prob = 0.01
noise_bias = 1e8
goal_logical_error_rate = bit_flip_prob/noise_bias


# SOLUTION ===

tasks = [
    sinter.Task(
        circuit=generate_rep_code_bit_flips(d, noise),
        json_metadata={'d': d, 'p': noise},
    )
    for d in [3,5,7]
    for noise in [bit_flip_prob]
]

collected_stats: List[sinter.TaskStats] = sinter.collect(
    num_workers=4,
    tasks=tasks,
    decoders=['pymatching'],
    max_shots=10**9,
    max_errors=1000,
)
# ===

In [ ]:
print([s.errors for s in collected_stats])
# If there are no zeros then continue else increase max shots or take only 3 and 5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

data = []
for s in collected_stats:
    if s.errors > 0:
        d = s.json_metadata['d']
        p_logical = s.errors / s.shots
        data.append((d, p_logical))

data.sort()
distances = np.array([d for d, p in data])
p_logical = np.array([p for d, p in data])


log_p = np.log10(p_logical)
slope_d, intercept, _, _, _ = linregress(distances, log_p)

target_p = 1e-10
required_d = (np.log10(target_p) - intercept) / slope_d
required_qubits = 2 * required_d - 1


fig, ax = plt.subplots(figsize=(10, 6))

qubits = 2 * distances - 1
ax.semilogy(qubits, p_logical, 'bo', label='Measured Data', markersize=8)


q_range = np.linspace(min(qubits)-1, np.ceil(required_qubits) + 1, 100)
d_range = (q_range + 1) / 2
p_range = 10**(slope_d * d_range + intercept)
ax.semilogy(q_range, p_range, 'r--', alpha=0.7, label=f'Linear Fit (Required d≈{required_d:.1f})')


ax.axhline(target_p, color='green', linestyle=':', label='Target 10^-10')
ax.axvline(required_qubits, color='orange', linestyle='--', label=f'Required Qubits ≈ {required_qubits:.1f}')


ax.set_title('Logical Error Rate vs. Number of Qubits', fontsize=14)
ax.set_xlabel('Number of Qubits ($2d - 1$)', fontsize=12)
ax.set_ylabel('Logical Error Rate ($P_L$)', fontsize=12)
ax.grid(True, which="both", ls="-", alpha=0.3)
ax.legend()

plt.show()

print(f"To reach 1e-10, you need d={int(np.ceil(required_d))} ({int(np.ceil(required_qubits))} qubits).")

In [ ]:
import pickle

# 1. Process your results into a simple dictionary of lists
results_to_store = {
    "distances": [s.json_metadata['d'] for s in collected_stats],
    "qubits": [2 * s.json_metadata['d'] - 1 for s in collected_stats],
    "logical_errors": [s.errors / s.shots for s in collected_stats],
    "raw_errors": [s.errors for s in collected_stats],
    "shots": [s.shots for s in collected_stats]
}

# 2. Perform the extrapolation to include in the file
fit_d = np.array([d for d, e in zip(results_to_store["distances"], results_to_store["raw_errors"]) if e > 0])
fit_p = np.array([p for p, e in zip(results_to_store["logical_errors"], results_to_store["raw_errors"]) if e > 0])

slope, intercept, _, _, _ = linregress(fit_d, np.log10(fit_p))
req_d = (np.log10(1e-10) - intercept) / slope
results_to_store["required_d_1e10"] = req_d
results_to_store["required_qubits_1e10"] = 2 * req_d - 1

# 3. Save using Pickle
with open('qec_data.pkl', 'wb') as f:
    pickle.dump(results_to_store, f)

print("Data stored in qec_data.pkl")